# Step 1.1: Initial Data Inspection

## Objective

The purpose of this step is to inspect the downloaded files before selecting a dataset for analysis.

The DataCo download contains multiple CSV files, but they may serve different purposes. One may contain operational supply-chain records, another may be a data dictionary, and another may contain unrelated system logs.

Before cleaning or analyzing the data, this inspection will determine:

- Which CSV files are available
- The size of each file
- Which encoding Python needs to read each file
- The number and names of the columns in each file
- Which file is relevant to shipment-exception analysis

Only the first five rows are read to inspect the file structure efficiently. Raw records are not displayed because the dataset may contain customer-related information.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

# Locate the project root reliably.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("\nCSV files found:", len(csv_files))

for file in csv_files:
    print("\n" + "=" * 80)
    print("FILE:", file.name)
    print("SIZE:", round(file.stat().st_size / 1_048_576, 2), "MB")

    for encoding in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            preview = pd.read_csv(
                file,
                encoding=encoding,
                nrows=5
            )

            print("ENCODING:", encoding)
            print("COLUMN COUNT:", len(preview.columns))
            print("COLUMNS:")

            for column in preview.columns:
                print("-", column)

            break

        except UnicodeDecodeError:
            continue

        except Exception as error:
            print("READ ERROR:", error)
            break

Project root: /Users/iffah/supply-chain-exception-control-tower
Data directory: /Users/iffah/supply-chain-exception-control-tower/data/raw

CSV files found: 3

FILE: DataCoSupplyChainDataset.csv
SIZE: 91.47 MB
ENCODING: latin-1
COLUMN COUNT: 53
COLUMNS:
- Type
- Days for shipping (real)
- Days for shipment (scheduled)
- Benefit per order
- Sales per customer
- Delivery Status
- Late_delivery_risk
- Category Id
- Category Name
- Customer City
- Customer Country
- Customer Email
- Customer Fname
- Customer Id
- Customer Lname
- Customer Password
- Customer Segment
- Customer State
- Customer Street
- Customer Zipcode
- Department Id
- Department Name
- Latitude
- Longitude
- Market
- Order City
- Order Country
- Order Customer Id
- order date (DateOrders)
- Order Id
- Order Item Cardprod Id
- Order Item Discount
- Order Item Discount Rate
- Order Item Id
- Order Item Product Price
- Order Item Profit Ratio
- Order Item Quantity
- Sales
- Order Item Total
- Order Profit Per Order
- Order 

## 1.2 Load the Main Dataset

The initial file inspection showed that `DataCoSupplyChainDataset.csv` is the main dataset for this project.

So I'll load the complete dataset and check:

- The number of rows
- The number of columns
- The data type of each column
- Whether columns contain missing values

In [2]:
# Create the path to the main dataset
main_data_file = DATA_DIR / "DataCoSupplyChainDataset.csv"

# Load the complete dataset
df = pd.read_csv(
    main_data_file,
    encoding="latin-1",
    low_memory=False
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 180519
Number of columns: 53


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

## 1.3 Identify the Dataset Grain

The dataset grain describes what one row represents.

Before calculating shipment-exception metrics, I need to determine whether each row represents:

- One order
- One shipment
- One product within an order
- Another type of transaction

An order may appear across multiple rows when it contains multiple products, so counting those rows as separate shipments would overstate the number of shipments and exceptions.

I will compare the number of unique order IDs and order-item IDs with the total number of rows.

In [5]:
# Count exact duplicate rows
duplicate_rows = df.duplicated().sum()

# Count unique identifiers
unique_orders = df["Order Id"].nunique()
unique_order_items = df["Order Item Id"].nunique()
unique_customers = df["Customer Id"].nunique()

print("Total rows:", len(df))
print("Exact duplicate rows:", duplicate_rows)
print("Unique order IDs:", unique_orders)
print("Unique order-item IDs:", unique_order_items)
print("Unique customer IDs:", unique_customers)

Total rows: 180519
Exact duplicate rows: 0
Unique order IDs: 65752
Unique order-item IDs: 180519
Unique customer IDs: 20652


In [6]:
# Count the number of rows belonging to each order
rows_per_order = df.groupby("Order Id").size()

# Count orders that appear in more than one row
orders_with_multiple_items = (rows_per_order > 1).sum()

print("Orders with multiple item rows:", orders_with_multiple_items)
print("Average number of item rows per order:", round(rows_per_order.mean(), 2))
print("Maximum number of item rows in one order:", rows_per_order.max())

Orders with multiple item rows: 45902
Average number of item rows per order: 2.75
Maximum number of item rows in one order: 5


### Initial Finding: What Does One Row Represent?

The dataset contains 180,519 rows but only 65,752 unique orders.

Each row has a unique order-item ID. This indicates that one row represents one order-item line within an order.

One order can appear in several rows when it contains multiple products. Therefore, counting dataset rows as separate deliveries would overstate the number of orders and delivery exceptions.

Before calculating delivery-exception KPIs, the analysis must decide how to create one record for each order.

## 1.4 Check Delivery Status Within Each Order

The dataset contains multiple item rows for some orders.

Before creating an order-level dataset, I need to check whether all item rows belonging to the same order have the same delivery status.

If every item within an order has the same delivery status, that status only needs to be counted once for the complete order.

In [7]:
# For each order, count the number of different delivery statuses
delivery_statuses_per_order = (
    df.groupby("Order Id")["Delivery Status"].nunique()
)

# Count orders that have more than one delivery status
orders_with_multiple_delivery_statuses = (
    delivery_statuses_per_order > 1
).sum()

print(
    "Orders with more than one delivery status:",
    orders_with_multiple_delivery_statuses
)

Orders with more than one delivery status: 0


### Finding

No orders contain more than one delivery status.

This means that all item rows belonging to the same order share the same delivery status. Therefore, delivery status should be counted once per order rather than once per item row.

This result supports using an order-level dataset for delivery-exception analysis. However, other shipping fields must also be checked before making the final decision.

In [8]:
# Shipping fields to check
shipping_fields = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Late_delivery_risk",
    "shipping date (DateOrders)",
    "Shipping Mode"
]

# Repeat the same consistency check for each field
for column in shipping_fields:
    values_per_order = df.groupby("Order Id")[column].nunique(
        dropna=False
    )

    orders_with_multiple_values = (
        values_per_order > 1
    ).sum()

    print(
        column,
        "- orders with more than one value:",
        orders_with_multiple_values
    )

Days for shipping (real) - orders with more than one value: 0
Days for shipment (scheduled) - orders with more than one value: 0
Late_delivery_risk - orders with more than one value: 0
shipping date (DateOrders) - orders with more than one value: 0
Shipping Mode - orders with more than one value: 0


### Finding: Shipping Information Is Consistent Within Orders

All shipping fields returned zero orders with inconsistent values.

This means that all item rows belonging to the same order share the same delivery status, shipping duration, late-delivery indicator, shipping date, and shipping mode.

Based on this evidence, the analysis can create one order-level delivery record for each unique `Order Id`. This will prevent the same delivery exception from being counted multiple times.

The dataset does not contain a separate shipment ID. Therefore, this project will use orders as the unit for delivery-exception analysis and clearly document this limitation.

## 1.5 Review the Data Dictionary

The dataset includes a data dictionary containing definitions for its columns.

Reviewing these definitions helps prevent incorrect assumptions about operational fields such as delivery status, shipping duration, and late-delivery risk.

In [9]:
# Load the data dictionary
dictionary_file = DATA_DIR / "DescriptionDataCoSupplyChain.csv"

data_dictionary = pd.read_csv(
    dictionary_file,
    encoding="utf-8"
)

# Remove extra spaces from the field names
data_dictionary["FIELDS"] = (
    data_dictionary["FIELDS"].str.strip()
)

# Fields most relevant to delivery-exception analysis
important_fields = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Order Id",
    "Order Status",
    "order date (DateOrders)",
    "Shipping date (DateOrders)",
    "Shipping Mode"
]

important_definitions = data_dictionary[
    data_dictionary["FIELDS"].isin(important_fields)
]

display(important_definitions)

,FIELDS,DESCRIPTION
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased product
5,Delivery Status,": Delivery status of orders: Advance shipping , Late delivery , Shipping canceled , Shipping on..."
6,Late_delivery_risk,": Categorical variable that indicates if sending is late (1), it is not late (0)."
28,order date (DateOrders),: Date on which the order is made
29,Order Id,: Order code
42,Order Status,": Order Status : COMPLETE , PENDING , CLOSED , PENDING_PAYMENT ,CANCELED , PROCESSING ,SUSPECTE..."
50,Shipping date (DateOrders),: Exact date and time of shipment
51,Shipping Mode,": The following shipping modes are presented : Standard Class , First Class , Second Class , Sa..."


### Check Order Status Within Each Order

Before including `Order Status` in the order-level dataset, I will check whether all item rows within the same order have the same order status.


In [10]:
# Count the number of different order statuses within each order
order_statuses_per_order = (
    df.groupby("Order Id")["Order Status"].nunique()
)

# Count orders that contain more than one order status
orders_with_multiple_order_statuses = (
    order_statuses_per_order > 1
).sum()

print(
    "Orders with more than one order status:",
    orders_with_multiple_order_statuses
)

Orders with more than one order status: 0


All item rows within the same order have the same `Order Status`. Therefore, this field can also be included once in the order-level dataset.


### Finding: Roles of the Operational Fields

`Delivery Status` is the clearest field for identifying delivery exceptions because it directly describes whether an order was delivered late, shipped on time, shipped early, or canceled.

`Late_delivery_risk` provides a simpler late-versus-not-late indicator. Its relationship with `Delivery Status` must be checked before using both fields.

The difference between actual and scheduled shipping days may help measure the severity of a late delivery.

`Order Status` may provide information about upstream order problems, such as cancellation, payment review, or suspected fraud. These statuses should not automatically be treated as shipment exceptions.

The shipping date is a time field rather than an exception field. It may be useful for analyzing trends over time.

## 1.6 Create an Order-Level Dataset

The raw dataset contains one row for each order-item line. However, delivery-exception analysis should count each order only once.

The consistency checks showed that all item rows within an order share the same delivery status, shipping duration, late-delivery indicator, shipping date, shipping mode, and order status.

I will create a separate DataFrame containing one row for each order. The original item-level DataFrame will remain unchanged.

In [11]:
# Select the order-level operational columns
order_columns = [
    "Order Id",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "shipping date (DateOrders)",
    "Shipping Mode",
    "Order Status"
]

# Keep one row for each order
orders_df = (
    df[order_columns]
    .drop_duplicates(subset="Order Id")
    .copy()
)

print("Rows in the original item-level dataset:", len(df))
print("Rows in the new order-level dataset:", len(orders_df))
print("Unique order IDs:", orders_df["Order Id"].nunique())

Rows in the original item-level dataset: 180519
Rows in the new order-level dataset: 65752
Unique order IDs: 65752


### Finding: Order-Level Dataset Created

The order-level dataset contains 65,752 rows and 65,752 unique order IDs. This confirms that each order now appears only once.

This dataset will be used for delivery-exception counts to prevent orders with multiple products from being counted several times.

## 1.7 Inspect Delivery Status

`Delivery Status` directly describes the delivery outcome of an order.

I will count the number and percentage of orders in each delivery-status category. The calculation will use the order-level dataset so that each order is counted only once.

In [12]:
# Count orders in each delivery-status category
delivery_status_counts = (
    orders_df["Delivery Status"]
    .value_counts(dropna=False)
)

# Convert the counts into a summary table
delivery_status_summary = delivery_status_counts.reset_index()

# Give the columns clear names
delivery_status_summary.columns = [
    "Delivery Status",
    "Order Count"
]

# Calculate each status as a percentage of all orders
delivery_status_summary["Percentage"] = (
    delivery_status_summary["Order Count"]
    / len(orders_df)
    * 100
).round(2)

display(delivery_status_summary)

,Delivery Status,Order Count,Percentage
0,Late delivery,36048,54.82
1,Advance shipping,15127,23.01
2,Shipping on time,11722,17.83
3,Shipping canceled,2855,4.34


### Initial Delivery-Status Finding

At the order level, 36,048 orders (54.82%) are classified as late deliveries. This is the most common delivery status in the dataset.

A further 2,855 orders (4.34%) are classified as shipping canceled.

Advance shipping represents 23.01% of orders, while 17.83% are classified as shipping on time.

These percentages describe the DataCo dataset and should not be presented as an industry benchmark.


## 1.8 Compare Delivery Status With Late-Delivery Risk

The dataset contains two fields that describe delivery lateness: `Delivery Status` and `Late_delivery_risk`.

I will compare these fields to check whether they classify late orders consistently.


In [13]:
# Compare delivery status with the late-delivery indicator
late_risk_comparison = pd.crosstab(
    orders_df["Delivery Status"],
    orders_df["Late_delivery_risk"],
    margins=True
)

# Add clearer labels
late_risk_comparison = late_risk_comparison.rename(
    columns={
        0: "Not Late (0)",
        1: "Late (1)",
        "All": "Total"
    },
    index={
        "All": "Total"
    }
)

display(late_risk_comparison)

Late_delivery_risk,Not Late (0),Late (1),Total
Delivery Status,,,
Advance shipping,15127,0,15127
Late delivery,0,36048,36048
Shipping canceled,2855,0,2855
Shipping on time,11722,0,11722
Total,29704,36048,65752


### Finding: The Two Late-Delivery Fields Are Consistent

All 36,048 orders classified as `Late delivery` have `Late_delivery_risk = 1`. All other delivery-status categories have `Late_delivery_risk = 0`.

This shows that the two fields are consistent. `Delivery Status` is more useful for this project because it provides the complete delivery outcome, while `Late_delivery_risk` only separates late and not-late orders.


## 1.9 Check Missing Values

Missing values are empty cells in the dataset.

I will count the missing values in each column and calculate their percentages. This check will show which columns may need to be removed or handled during data cleaning. No data will be changed in this step.


In [14]:
# Count missing values in every column
missing_values = df.isna().sum()

# Calculate the percentage of missing values
missing_percentage = (
    missing_values / len(df) * 100
).round(4)

# Combine the results into one table
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

# Show only columns that contain missing values
missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

display(missing_summary)


,Missing Values,Missing Percentage
Product Description,180519,100.0000
Order Zipcode,155679,86.2397
Customer Lname,8,0.0044
Customer Zipcode,3,0.0017


### Step 1 Conclusion

The DataCo download contains three CSV files. `DataCoSupplyChainDataset.csv` is the main operational dataset, `DescriptionDataCoSupplyChain.csv` is the data dictionary, and `tokenized_access_logs.csv` is not required for this shipment-exception project.

The main dataset contains 180,519 order-item rows and 65,752 unique orders. Shipping information is consistent within each order, so one order-level record was created for each `Order Id`. Because the dataset does not provide a separate Shipment ID, Order ID will be used as a proxy for a delivery record and this will be documented as a limitation.

At the order level, 54.82% of orders are classified as late deliveries and 4.34% as shipping canceled. `Delivery Status` and `Late_delivery_risk` classify late orders consistently.

The missing-value check identifies columns that require attention during data cleaning. No values were removed or changed during this initial inspection.
